# Ingredient match debug — top-20 candidates with score breakdown

Uses the same staged matcher as [`food_mvp_recipe_matching.ipynb`](food_mvp_recipe_matching.ipynb) (`food_4macro` + cached embeddings).

Each query shows **ingredient-parser-nlp** fields (`quantity`, `unit`, `name`, `size`, `preparation`, …) then top-20 USDA matches with full **description** text (no truncation).

Requires cache under `scratch/recipe_matching_10k/` (run `ingredient_query_cache.py` or the main notebook first).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))

from ingredient_match_staged import (
    StagedMatchConfig,
    StagedFoodIndex,
    match_query_top_k,
    query_from_parsed_row,
)
from ingredient_query_cache import (
    embed_adhoc_recipe_queries,
    ensure_hf_token,
    load_or_build_food_artifacts,
)
from load_food_4macro import load_food_4macro
from parse_recipe_ingredient import PARSE_FIELDS
from progress_utils import force_std_tqdm
from recipe_match_cache import DEFAULT_CACHE_DIR

force_std_tqdm()
ensure_hf_token()

# Show full USDA descriptions (no "..." truncation in notebook output)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

WORK_DIR = DEFAULT_CACHE_DIR
MATCH_CONFIG = StagedMatchConfig()
TOP_K = 20

# ingredient-parser-nlp fields + pipeline extras
PARSED_DISPLAY_COLS = [
    "ingredient",
    *PARSE_FIELDS,
    "dequantified",
    "prep_used_unprepared",
]

/Users/danielcosta/.pyenv/versions/3.11.11/lib/python3.11/importlib/__init__.py:126: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  return _bootstrap._gcd_import(name[level:], package, level)


In [2]:
food_4macro_raw = load_food_4macro()
_, food_name_emb, food_prep_emb, food_dequant_emb, food_meta = load_or_build_food_artifacts(
    food_4macro_raw,
    WORK_DIR,
)
food_index = StagedFoodIndex.from_catalog(
    food_4macro_raw,
    name_embeddings=food_name_emb,
    prep_embeddings=food_prep_emb,
    dequant_embeddings=food_dequant_emb,
    config=MATCH_CONFIG,
    show_progress=False,
)
print(f"food index: {len(food_index.candidates):,} candidates")

Loaded cached food_4macro parse + embeddings (96,996 rows) → /Users/danielcosta/Berkeley/Capstone/scratch/recipe_matching_10k
food index: 96,996 candidates


In [3]:
examples = [
    "4 chicken breasts, skin removed",
    "1 Tbsp vanilla",
    "2 (16 oz.) cans lima beans",
]

parsed, name_emb, prep_emb, dequant_emb = embed_adhoc_recipe_queries(examples, WORK_DIR)

# ingredient-parser-nlp output (see scripts/parse_recipe_ingredient.py)
cols = [c for c in PARSED_DISPLAY_COLS if c in parsed.columns]
display(parsed[cols].T)  # transposed: one column per example, easier to read

Using HF_TOKEN from .env for Hugging Face Hub downloads.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6921.18it/s]


Recipe prep: 2 lines use 'unprepared' proxy; 1 embed parsed preparation


,ingredient,name,preparation,dequantified,prep_used_unprepared
0,"4 chicken breasts, skin removed",chicken breasts,skin removed,"chicken breasts, skin removed",False
1,1 Tbsp vanilla,vanilla,,vanilla,True
2,2 (16 oz.) cans lima beans,lima beans,,lima beans,True


In [4]:
SCORE_COLS = [
    "rank",
    "fdc_id",
    "description",
    "match_score",
    "match_margin",
    "base_score",
    "prep_score",
    "default_bonus",
    "modifier_penalty",
    "name_lex_score",
    "name_sem_score",
    "dequant_lex_score",
    "dequant_sem_score",
    "name_channel_score",
    "dequant_channel_score",
    "prep_sem_score",
    "prep_lex_score",
    "prep_rules_score",
    "prep_default_component",
    "fallback_reason",
]


def display_full(df: pd.DataFrame) -> None:
    """Render DataFrame with untruncated text columns."""
    display(
        df.style.set_properties(
            subset=[c for c in df.columns if df[c].dtype == object],
            **{"white-space": "pre-wrap", "text-align": "left"},
        )
    )


def show_parser_fields(i: int) -> None:
    row = parsed.iloc[i]
    fields = [c for c in PARSED_DISPLAY_COLS if c in parsed.columns and c != "ingredient"]
    summary = pd.Series({c: row[c] for c in fields}, name=examples[i])
    display_full(summary.to_frame())


def show_top_k(i: int) -> pd.DataFrame:
    row = parsed.iloc[i]
    q = query_from_parsed_row(row, name_emb[i], prep_emb[i], dequant_emb[i])
    top = match_query_top_k(q, food_index, top_k=TOP_K, score_all_stage1=True)
    cols = [c for c in SCORE_COLS if c in top.columns]
    return top[cols]


for i, text in enumerate(examples):
    print("=" * 80)
    print(f"Query: {text}")
    print("\nParser fields (ingredient-parser-nlp):")
    show_parser_fields(i)
    print(f"\nTop {TOP_K} candidates:")
    display_full(show_top_k(i))

Query: 4 chicken breasts, skin removed
  parsed name='chicken breasts' prep='skin removed' dequant='chicken breasts, skin removed' unprepared_proxy=False


,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,171474,"Chicken, broilers or fryers, breast, meat and ...",0.3911,0.0621,0.6002,0.5637,1.0,0.24,0.5964,0.6398,0.6141,0.4871,0.6051,0.5887,0.2843,0.5,1.0,1.0,exact_prep
1,2,172394,"Chicken, roasting, meat and skin and giblets a...",0.3290,0.0233,0.5775,0.3746,1.0,0.24,0.5964,0.7707,0.4332,0.5272,0.6313,0.4520,0.3115,0.5,0.0,1.0,base_only
2,3,172378,"Chicken, broilers or fryers, leg, meat and ski...",0.3057,0.0002,0.5458,0.3637,1.0,0.24,0.5964,0.5713,0.4379,0.4456,0.5914,0.4394,0.2843,0.5,0.0,1.0,base_only
3,4,172390,"Chicken, broilers or fryers, wing, meat and sk...",0.3055,0.0000,0.5454,0.3637,1.0,0.24,0.5964,0.5713,0.4364,0.4456,0.5914,0.4382,0.2843,0.5,0.0,1.0,base_only
4,5,172382,"Chicken, broilers or fryers, neck, meat and sk...",0.3055,0.0000,0.5454,0.3637,1.0,0.24,0.5964,0.5713,0.4364,0.4456,0.5914,0.4382,0.2843,0.5,0.0,1.0,base_only
5,6,172373,"Chicken, broilers or fryers, drumstick, meat a...",0.3055,0.0002,0.5454,0.3637,1.0,0.24,0.5964,0.5610,0.4434,0.4413,0.5893,0.4430,0.2843,0.5,0.0,1.0,base_only
6,7,172385,"Chicken, broilers or fryers, thigh, meat and s...",0.3053,0.0185,0.5451,0.3637,1.0,0.24,0.5964,0.5713,0.4349,0.4456,0.5914,0.4371,0.2843,0.5,0.0,1.0,base_only
7,8,171075,"Chicken, broilers or fryers, breast, meat and ...",0.2868,0.0128,0.6046,0.4552,0.2,0.24,0.5964,0.6899,0.6140,0.4437,0.6151,0.5799,0.2130,0.5,1.0,0.2,base_only
8,9,171076,"Chicken, broilers or fryers, breast, meat and ...",0.2740,0.0101,0.6000,0.4159,0.2,0.24,0.5964,0.6899,0.6028,0.4131,0.6151,0.5649,0.1147,0.5,1.0,0.2,base_only
9,10,171123,"Chicken, broilers or fryers, rotisserie, origi...",0.2639,0.0599,0.5732,0.4455,0.2,0.24,0.5964,0.5630,0.5694,0.3951,0.5898,0.5345,0.1887,0.5,1.0,0.2,base_only


Query: 1 Tbsp vanilla
  parsed name='vanilla' prep='' dequant='vanilla' unprepared_proxy=True


,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,1110492,"VANILLA BAR, VANILLA",0.6282,0.0407,0.6602,0.4362,0.9,0.0,0.6250,0.8011,0.6250,0.8011,0.6602,0.6602,0.2404,0.5,0.5,0.9,neutral_default
1,2,1122218,"HEAVENLY VANILLA PREMIUM ICE CREAM, HEAVENLY V...",0.5875,0.0002,0.5977,0.4362,0.9,0.0,0.5800,0.6683,0.5800,0.6683,0.5977,0.5977,0.2404,0.5,0.5,0.9,neutral_default
2,3,356469,VANILLA BEAN ICE CREAM,0.5873,0.0018,0.5974,0.4362,0.9,0.0,0.5875,0.6368,0.5875,0.6368,0.5974,0.5974,0.2404,0.5,0.5,0.9,neutral_default
3,4,1110260,"HOMEMADE VANILLA PREMIUM ICE CREAM, HOMEMADE V...",0.5855,0.0042,0.5946,0.4362,0.9,0.0,0.5800,0.6528,0.5800,0.6528,0.5946,0.5946,0.2404,0.5,0.5,0.9,neutral_default
4,5,1110259,"FRENCH VANILLA PREMIUM ICE CREAM, FRENCH VANILLA",0.5813,0.0007,0.5881,0.4362,0.9,0.0,0.5800,0.6206,0.5800,0.6206,0.5881,0.5881,0.2404,0.5,0.5,0.9,neutral_default
5,6,403484,VANILLA CUPCAKE & CAKE,0.5806,0.0007,0.5871,0.4362,0.9,0.0,0.6000,0.5354,0.6000,0.5354,0.5871,0.5871,0.2404,0.5,0.5,0.9,neutral_default
6,7,1122226,"VANILLA CARAMEL COFFEE CREAMER, VANILLA CARAMEL",0.5799,0.0002,0.5859,0.4362,0.9,0.0,0.5875,0.5793,0.5875,0.5793,0.5859,0.5859,0.2404,0.5,0.5,0.9,neutral_default
7,8,1122220,"VANILLA BEAN PREMIUM ICE CREAM, VANILLA BEAN",0.5797,0.0013,0.5856,0.4362,0.9,0.0,0.5800,0.6078,0.5800,0.6078,0.5856,0.5856,0.2404,0.5,0.5,0.9,neutral_default
8,9,1110263,"VANILLA THE AUTHENTIC ITALIAN WAFFLE COOKIE, V...",0.5784,0.0042,0.5836,0.4362,0.9,0.0,0.5800,0.5980,0.5800,0.5980,0.5836,0.5836,0.2404,0.5,0.5,0.9,neutral_default
9,10,1122187,"FRENCH VANILLA ALMOND + COCONUT CREAMER, FRENC...",0.5742,0.0064,0.5771,0.4362,0.9,0.0,0.5800,0.5657,0.5800,0.5657,0.5771,0.5771,0.2404,0.5,0.5,0.9,neutral_default


Query: 2 (16 oz.) cans lima beans
  parsed name='lima beans' prep='' dequant='lima beans' unprepared_proxy=True


,rank,fdc_id,description,match_score,match_margin,base_score,prep_score,default_bonus,modifier_penalty,name_lex_score,name_sem_score,dequant_lex_score,dequant_sem_score,name_channel_score,dequant_channel_score,prep_sem_score,prep_lex_score,prep_rules_score,prep_default_component,fallback_reason
0,1,174252,"Lima beans, large, mature seeds, raw",0.7275,0.0183,0.7810,0.4792,1.0,0.0,0.8263,0.7371,0.7174,0.7152,0.8085,0.7170,0.3229,0.5,0.5,1.0,neutral_default
1,2,1117820,LIMA BEANS,0.7092,0.0010,0.7849,0.4362,0.9,0.0,0.7316,1.0000,0.7300,1.0000,0.7853,0.7840,0.2404,0.5,0.5,0.9,neutral_default
2,3,174255,"Lima beans, thin seeded (baby), mature seeds, raw",0.7082,0.0394,0.7689,0.4334,1.0,0.0,0.8263,0.7408,0.6689,0.6996,0.8092,0.6750,0.2085,0.5,0.5,1.0,neutral_default
3,4,1136235,BABY LIMA BEANS,0.6688,0.0012,0.7227,0.4362,0.9,0.0,0.6750,0.9146,0.6740,0.9146,0.7229,0.7221,0.2404,0.5,0.5,0.9,neutral_default
4,5,390919,GREEN LIMA BEANS,0.6676,0.0057,0.7209,0.4362,0.9,0.0,0.6740,0.9097,0.6731,0.9097,0.7211,0.7204,0.2404,0.5,0.5,0.9,neutral_default
5,6,1135568,LARGE LIMA BEANS,0.6619,0.0202,0.7121,0.4362,0.9,0.0,0.6740,0.8655,0.6731,0.8655,0.7123,0.7116,0.2404,0.5,0.5,0.9,neutral_default
6,7,390890,FRESH CUT LIMA BEANS,0.6417,0.0014,0.6811,0.4362,0.9,0.0,0.6457,0.8235,0.6450,0.8235,0.6813,0.6807,0.2404,0.5,0.5,0.9,neutral_default
7,8,1120032,DELUXE TINY LIMA BEANS,0.6403,0.0040,0.6788,0.4362,0.9,0.0,0.6444,0.8175,0.6438,0.8175,0.6790,0.6785,0.2404,0.5,0.5,0.9,neutral_default
8,9,377806,SEASONED GREEN LIMA BEANS,0.6363,0.0050,0.6727,0.4362,0.9,0.0,0.6426,0.7933,0.6421,0.7933,0.6728,0.6724,0.2404,0.5,0.5,0.9,neutral_default
9,10,169318,"Lima beans, immature seeds, frozen, baby, cook...",0.6313,0.0130,0.8046,0.3530,0.2,0.0,0.8263,1.0000,0.6498,0.7661,0.8611,0.6730,0.2074,0.5,0.5,0.2,base_only
